# tcpyVPI: ERA5 vPI and GPIv example read+calculate+plot from monthly and hourly data using built-in wrapper function

This notebook demonstrates how to use the tcpyVPI package to compute
ventilated Potential Intensity (vPI) and Genesis Potential Index (GPIv)
from ERA5 monthly and hourly reanalysis data using built-in wrapper functions in the package.

IMPORTANT: this notebook reads ERA5 data remotely via the NCAR THREDDS server. Sometimes THREDDS throws an "NetCDF: DAP server error" when reading the data -- it seems to be somewhat random whether/when it happens and is not due to this notebook. If it happens, run it again.

**Cite this package:**  
Chavas, D. Sanchez, J. O., and A. Kruskie (2026). *tcpyVPI*. https://doi.org/10.5281/zenodo.19319996

[![PyPI version](https://img.shields.io/pypi/v/tcpyVPI.svg)](https://pypi.org/project/tcpyVPI/)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.19319996.svg)](https://doi.org/10.5281/zenodo.19319996)


**Key Features:**
- Load ERA5 monthly and hourly data via THREDDS
- Compute vPI and GPIv (Chavas et al 2025) for a specific year+month

**Author:** Jose Ocegueda Sanchez (2025). Edited for this file: Dan Chavas

In [ ]:
!pip install -q tcpyPI "tcpyVPI>=1.2.0" cartopy


In [ ]:
# Import required packages
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Import tcvpigpiv
from tcpyVPI import (
    run_vpigpiv,
    run_vpigpiv_hourly,
    load_era5_data,
    compute_gpiv_from_dataset,
)

## 1. Monthly ERA5 vPI and GPIv

In [ ]:
# Compute monthly mean GPIv for September 2022
year = 2022
month = 9

results_monthly = run_vpigpiv(year, month, data_source='monthly', plot=True)
print(f"\nVariables computed: {list(results_monthly.data_vars)}")

Loading ERA5 monthly mean data for 2022-09...
  Loading SSTK...
  Loading SP...
  Loading T...
  Loading Q...


OSError: [Errno -70] NetCDF: DAP server error: 'https://thredds.rda.ucar.edu/thredds/dodsC/files/g/d633001_nc/e5.moda.an.pl/2022/e5.moda.an.pl.128_133_q.ll025sc.2022010100_2022120100.nc'

## 2. Hourly ERA5 vPI and GPIv

In [ ]:
# Compute hourly GPIv for August 15, 2020 at 12Z
year = 2020
month = 8
day = 15
hour = 12

results_hourly = run_vpigpiv_hourly(year, month, day, hour=hour, plot=True)
print(f"\nVariables computed: {list(results_hourly.data_vars)}")

Loading ERA5 hourly data for 2020-08-15 hour 12...
  Loading SSTK (surface, monthly file)...
  Loading SP (surface, monthly file)...


OSError: [Errno -70] NetCDF: DAP server error: 'https://thredds.rda.ucar.edu/thredds/dodsC/files/g/d633000/e5.oper.an.sfc/202008/e5.oper.an.sfc.128_134_sp.ll025sc.2020080100_2020083123.nc'

## 3. Optional: changing the configuration

Sections 1 and 2 use the defaults, which reproduce Chavas et al. (2025). Since
v1.2.0 the choices that used to be hardcoded can be passed as keyword arguments
instead, for sensitivity testing. Every default below is the published value, so
running this cell unchanged gives the same answer as section 1.

In [ ]:
PARAMS = dict(
    # --- pressure levels -------------------------------------------------
    shear_p_top   = 200.0,     # hPa, top of the bulk shear layer
    shear_p_bot   = 850.0,     # hPa, bottom of the bulk shear layer
    chi_p_mid     = 600.0,     # hPa, mid-level for the entropy deficit
    vort_level    = 850.0,     # hPa, level of the relative vorticity
    # --- thresholds ------------------------------------------------------
    vort_cap      = 3.7e-5,    # s^-1, cap on absolute vorticity
    VI_max        = 0.145,     # ventilation index above which vPI = 0
    # --- GPIv fit --------------------------------------------------------
    gpiv_exponent = 4.90,      # GPIv = (102.1 * vPI * eta_c) ** exponent
    # --- potential intensity (passed through to tcpyPI) ------------------
    CKCD          = 0.9,       # ratio C_k / C_d
    ascent_flag   = 0,         # 0 = reversible, 1 = pseudo-adiabatic
    diss_flag     = 1,         # 1 = dissipative heating on, 0 = off
    ptop          = 50.0,      # hPa, sounding above this is ignored
)

results_custom = run_vpigpiv(2022, 9, data_source='monthly',
                             plot=False, verbose=False, **PARAMS)

# Untouched, this must match section 1 exactly.
same = np.allclose(np.nan_to_num(results_custom['vPI'].values, nan=-9e9),
                   np.nan_to_num(results_monthly['vPI'].values, nan=-9e9))
print("matches the defaults:", same)
print("vPI mean:", float(results_custom['vPI'].mean()))

Two things to know:

- **Pressure levels are selected exactly, not by nearest neighbour.** If you ask
  for a level the dataset does not contain you get a `KeyError` listing what is
  available; interpolate the dataset onto that level first.
- **`gpiv_exponent` is tied to the normalising constant 102.1**, which was
  calibrated jointly with the default 4.90 to match the observed global mean
  genesis count. Changing the exponent alone leaves GPIv un-normalised: spatial
  patterns stay meaningful, absolute values do not.

A worked default-vs-custom comparison, with a difference map, is in
`github_tests/tcpyVPI_ERA5_from_github.ipynb`.